
# Notebook 01 — Factibilidad de STU-Net-S congelada en RTX 3050 Ti

**Objetivo único:** comprobar que una STU-Net-S preentrenada puede ejecutar inferencia 3D estable sobre **un estudio de TC** en una laptop con RTX 3050 Ti, sin fine-tuning.

Este notebook:

- usa la implementación oficial de STU-Net basada en nnU-Net v1;
- descarga y organiza el checkpoint oficial `small_ep4k`;
- acepta un volumen NIfTI o una carpeta DICOM;
- ejecuta inferencia con `batch=1`, precisión mixta, sin TTA y modo rápido;
- registra tiempo, utilización, temperatura y pico de VRAM;
- verifica que aparezcan las etiquetas `kidney_left` y `kidney_right`;
- guarda un manifiesto reproducible del experimento.

**Este notebook todavía no implementa TurboConv.** Su función es cerrar primero la pregunta de factibilidad del modelo 3D congelado. El siguiente notebook reproducirá PTQ y TurboConv sobre sus pesos.

## Entorno recomendado

La ruta con menos fricción es **WSL2 o Linux**, con un entorno separado:

```bash
conda create -n stunet39 python=3.9 -y
conda activate stunet39

pip install torch==1.10.2+cu113 torchvision==0.11.3+cu113 \
  --extra-index-url https://download.pytorch.org/whl/cu113

pip install jupyterlab ipykernel
python -m ipykernel install --user --name stunet39 --display-name "Python 3 (STU-Net)"
```

Después abre este notebook con el kernel **Python 3 (STU-Net)**.

> El código oficial declara `torch==1.10` y `nnUNet==1.7.0`. Versiones más recientes pueden funcionar, pero este primer piloto no debe gastar tiempo persiguiendo incompatibilidades evitables.


In [ ]:

# 1. Diagnóstico del entorno
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

def run_text(cmd: list[str]) -> str:
    completed = subprocess.run(
        cmd,
        check=False,
        capture_output=True,
        text=True,
    )
    return (completed.stdout or completed.stderr).strip()

print("Python:", sys.version.replace("\n", " "))
print("Sistema:", platform.platform())
print("Ejecutable:", sys.executable)
print("nvidia-smi:", shutil.which("nvidia-smi"))

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    print("CUDA de PyTorch:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM total: {props.total_memory / 1024**3:.2f} GiB")
except Exception as exc:
    print("PyTorch no pudo importarse:", repr(exc))

if shutil.which("nvidia-smi"):
    print("\nResumen de la GPU:")
    print(run_text([
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,memory.used,temperature.gpu",
        "--format=csv,noheader",
    ]))



## 2. Configuración

Edita únicamente esta celda antes de ejecutar el resto.

- Usa `INPUT_MODE = "nifti"` cuando ya tengas un `.nii` o `.nii.gz`.
- Usa `INPUT_MODE = "dicom"` cuando tengas una carpeta con cortes DICOM.
- En Windows puro, una ruta puede escribirse como `Path(r"D:\datos\caso")`.
- En WSL2, la misma unidad suele verse como `Path("/mnt/d/datos/caso")`.

El primer intento debe usar **un solo caso**, preferentemente una TC abdominal completa y con valores HU originales.


In [ ]:

# 2. Configuración editable
from pathlib import Path

PROJECT_ROOT = Path.home() / "stunet_turboconv_pilot"
CASE_ID = "tcga_kirc_pilot_001"

# "nifti" o "dicom"
INPUT_MODE = "nifti"

# Edita una de estas rutas:
SOURCE_NIFTI = Path(r"/ruta/al/volumen_tc.nii.gz")
SOURCE_DICOM_DIR = Path(r"/ruta/a/la/carpeta_dicom")

# Si la carpeta DICOM contiene varias series, deja None para elegir
# automáticamente la serie con mayor número de cortes.
DICOM_SERIES_UID = None

# Controles de ejecución
RUN_SETUP = False        # True solo la primera vez
RUN_PREPARE_INPUT = False
RUN_INFERENCE = False

# El piloto debe usar la variante pequeña.
TRAINER = "STUNetTrainer_small"
CHECKPOINT_NAME = "small_ep4k"
TASK_ID = "101"
MODEL = "3d_fullres"

# Rutas internas: normalmente no se editan.
REPO_DIR = PROJECT_ROOT / "STU-Net"
NNUNET_DIR = REPO_DIR / "nnUNet-1.7.1"

ENV_ROOT = PROJECT_ROOT / "nnunet_env"
RAW_BASE = ENV_ROOT / "raw"
PREPROCESSED = ENV_ROOT / "preprocessed"
RESULTS_FOLDER = ENV_ROOT / "results"

INPUT_DIR = PROJECT_ROOT / "pilot_input"
OUTPUT_DIR = PROJECT_ROOT / "pilot_output"
LOG_DIR = PROJECT_ROOT / "logs"

MODEL_DIR = (
    RESULTS_FOLDER
    / "nnUNet"
    / "3d_fullres"
    / "Task101_TotalSegmentator"
    / "STUNetTrainer_small__nnUNetPlansv2.1"
)
FOLD_DIR = MODEL_DIR / "fold_0"

PREPARED_INPUT = INPUT_DIR / f"{CASE_ID}_0000.nii.gz"
PREDICTION_PATH = OUTPUT_DIR / f"{CASE_ID}.nii.gz"

for directory in [
    PROJECT_ROOT, ENV_ROOT, RAW_BASE, PREPROCESSED, RESULTS_FOLDER,
    INPUT_DIR, OUTPUT_DIR, LOG_DIR, FOLD_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Proyecto:", PROJECT_ROOT)
print("Entrada preparada:", PREPARED_INPUT)
print("Predicción esperada:", PREDICTION_PATH)



## 3. Instalación reproducible y descarga del checkpoint

Esta celda no reinstala PyTorch. Instala las dependencias auxiliares, clona el repositorio oficial, instala el fork de nnU-Net incluido por STU-Net y descarga:

- `small_ep4k.model`;
- `small_ep4k.model.pkl`;
- `plans.pkl`;
- `label_orders.json`.

Después de la primera instalación, **reinicia el kernel** si aparece un error de importación.


In [ ]:

# 3. Setup oficial de STU-Net-S
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

OFFICIAL_REPO = "https://github.com/uni-medical/STU-Net.git"
SMALL_MODEL_GDRIVE_ID = "1HReH6dDrEuXgHPrsw7OrHSjvEUF3f4mv"

RAW_BASE_URL = "https://raw.githubusercontent.com/uni-medical/STU-Net/main"
PLAN_URL = f"{RAW_BASE_URL}/plan_files/plans.pkl"
MODEL_PKL_URL = f"{RAW_BASE_URL}/plan_files/small_ep4k.model.pkl"
LABELS_URL = f"{RAW_BASE_URL}/label_orders.json"

def checked_run(cmd: list[str], cwd: Path | None = None) -> None:
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

if not RUN_SETUP:
    print("RUN_SETUP=False. Cambia a True solo para la instalación inicial.")
else:
    checked_run([
        sys.executable, "-m", "pip", "install",
        "gdown", "nibabel", "SimpleITK", "pandas",
        "psutil", "matplotlib", "tqdm", "torchinfo"
    ])

    if not REPO_DIR.exists():
        checked_run(["git", "clone", OFFICIAL_REPO, str(REPO_DIR)])
    else:
        print("Repositorio ya presente:", REPO_DIR)

    if not NNUNET_DIR.exists():
        raise FileNotFoundError(
            f"No se encontró el nnU-Net v1 incluido en STU-Net: {NNUNET_DIR}"
        )

    checked_run([sys.executable, "-m", "pip", "install", "-e", str(NNUNET_DIR)])

    import gdown

    checkpoint_path = FOLD_DIR / f"{CHECKPOINT_NAME}.model"
    checkpoint_pkl_path = FOLD_DIR / f"{CHECKPOINT_NAME}.model.pkl"
    plans_path = MODEL_DIR / "plans.pkl"
    labels_path = PROJECT_ROOT / "label_orders.json"

    if not checkpoint_path.exists():
        gdown.download(
            id=SMALL_MODEL_GDRIVE_ID,
            output=str(checkpoint_path),
            quiet=False,
        )

    if not checkpoint_pkl_path.exists():
        urllib.request.urlretrieve(MODEL_PKL_URL, checkpoint_pkl_path)

    if not plans_path.exists():
        urllib.request.urlretrieve(PLAN_URL, plans_path)

    if not labels_path.exists():
        urllib.request.urlretrieve(LABELS_URL, labels_path)

    print("\nSetup terminado.")
    print("Reinicia el kernel si nnU-Net no puede importarse en la siguiente celda.")


In [ ]:

# 4. Verificación de instalación, variables de entorno y archivos
import os
import shutil
from pathlib import Path

os.environ["nnUNet_raw_data_base"] = str(RAW_BASE)
os.environ["nnUNet_preprocessed"] = str(PREPROCESSED)
os.environ["RESULTS_FOLDER"] = str(RESULTS_FOLDER)

required_files = {
    "checkpoint": FOLD_DIR / f"{CHECKPOINT_NAME}.model",
    "checkpoint_metadata": FOLD_DIR / f"{CHECKPOINT_NAME}.model.pkl",
    "plans": MODEL_DIR / "plans.pkl",
    "labels": PROJECT_ROOT / "label_orders.json",
}

print("Variables nnU-Net:")
for name in ["nnUNet_raw_data_base", "nnUNet_preprocessed", "RESULTS_FOLDER"]:
    print(f"  {name}={os.environ[name]}")

print("\nArchivos:")
missing = []
for name, path in required_files.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1024**2 if exists else 0.0
    print(f"  {name:22s} | {'OK' if exists else 'FALTA':5s} | {size_mb:8.2f} MiB | {path}")
    if not exists:
        missing.append(path)

predict_executable = shutil.which("nnUNet_predict")
print("\nnnUNet_predict:", predict_executable)

if missing:
    print("\nFaltan archivos. Ejecuta primero la celda de setup con RUN_SETUP=True.")
elif predict_executable is None:
    print("\nNo se encontró nnUNet_predict. Reinicia el kernel o revisa la instalación editable.")
else:
    print("\nInstalación lista para preparar el caso.")



## 5. Preparación del caso

La conversión DICOM:

1. enumera las series disponibles;
2. selecciona la serie indicada o la que tenga más cortes;
3. conserva origen, orientación y espaciado mediante SimpleITK;
4. escribe el archivo con el sufijo `_0000.nii.gz` requerido por nnU-Net.

No se aplica recorte manual ni normalización fuera del preprocesamiento oficial del modelo.


In [ ]:

# 5. Preparar un único estudio
from __future__ import annotations

import shutil
from pathlib import Path

def convert_dicom_series(
    dicom_dir: Path,
    output_path: Path,
    series_uid: str | None = None,
) -> dict:
    import SimpleITK as sitk

    if not dicom_dir.exists():
        raise FileNotFoundError(dicom_dir)

    reader = sitk.ImageSeriesReader()
    series_ids = reader.GetGDCMSeriesIDs(str(dicom_dir))
    if not series_ids:
        raise RuntimeError(f"No se encontraron series DICOM en {dicom_dir}")

    series_summary = []
    for uid in series_ids:
        filenames = reader.GetGDCMSeriesFileNames(str(dicom_dir), uid)
        series_summary.append((uid, len(filenames)))

    if series_uid is None:
        selected_uid, n_files = max(series_summary, key=lambda item: item[1])
    else:
        matches = [item for item in series_summary if item[0] == series_uid]
        if not matches:
            raise ValueError(
                f"La serie {series_uid!r} no existe. Disponibles: {series_summary}"
            )
        selected_uid, n_files = matches[0]

    filenames = reader.GetGDCMSeriesFileNames(str(dicom_dir), selected_uid)
    reader.SetFileNames(filenames)
    image = reader.Execute()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(image, str(output_path), useCompression=True)

    return {
        "selected_series_uid": selected_uid,
        "number_of_slices": int(n_files),
        "available_series": [
            {"uid": uid, "number_of_files": int(count)}
            for uid, count in series_summary
        ],
        "size_xyz": list(map(int, image.GetSize())),
        "spacing_xyz_mm": list(map(float, image.GetSpacing())),
        "origin_xyz": list(map(float, image.GetOrigin())),
        "direction": list(map(float, image.GetDirection())),
    }

preparation_metadata = {}

if not RUN_PREPARE_INPUT:
    print("RUN_PREPARE_INPUT=False. Edita la ruta de entrada y cambia a True.")
else:
    if PREPARED_INPUT.exists():
        PREPARED_INPUT.unlink()

    if INPUT_MODE.lower() == "nifti":
        if not SOURCE_NIFTI.exists():
            raise FileNotFoundError(SOURCE_NIFTI)
        shutil.copy2(SOURCE_NIFTI, PREPARED_INPUT)
        preparation_metadata = {
            "input_mode": "nifti",
            "source": str(SOURCE_NIFTI.resolve()),
        }

    elif INPUT_MODE.lower() == "dicom":
        preparation_metadata = convert_dicom_series(
            SOURCE_DICOM_DIR,
            PREPARED_INPUT,
            DICOM_SERIES_UID,
        )
        preparation_metadata.update({
            "input_mode": "dicom",
            "source": str(SOURCE_DICOM_DIR.resolve()),
        })

    else:
        raise ValueError("INPUT_MODE debe ser 'nifti' o 'dicom'.")

    print("Entrada preparada:", PREPARED_INPUT)
    print(json.dumps(preparation_metadata, indent=2, ensure_ascii=False))


In [ ]:

# 6. Control de calidad mínimo del NIfTI
from __future__ import annotations

import json
import numpy as np

if not PREPARED_INPUT.exists():
    print("Todavía no existe la entrada preparada:", PREPARED_INPUT)
else:
    import nibabel as nib

    image = nib.load(str(PREPARED_INPUT))
    data_proxy = image.dataobj

    # Muestreo central para evitar cargar dos veces un volumen grande.
    shape = image.shape
    z_mid = shape[2] // 2
    center_slice = np.asarray(data_proxy[:, :, z_mid], dtype=np.float32)

    finite = np.isfinite(center_slice)
    finite_values = center_slice[finite]

    qc = {
        "shape": list(map(int, shape)),
        "voxel_spacing_mm": [float(x) for x in image.header.get_zooms()[:3]],
        "dtype_on_disk": str(image.get_data_dtype()),
        "center_slice_finite_fraction": float(finite.mean()),
        "center_slice_min": float(finite_values.min()) if finite_values.size else None,
        "center_slice_max": float(finite_values.max()) if finite_values.size else None,
        "center_slice_percentiles": {
            str(p): float(np.percentile(finite_values, p))
            for p in [0.5, 1, 50, 99, 99.5]
        } if finite_values.size else {},
        "affine": image.affine.tolist(),
    }

    print(json.dumps(qc, indent=2, ensure_ascii=False))

    if not finite.all():
        raise ValueError("La rebanada central contiene NaN o Inf.")

    if finite_values.size and (
        np.percentile(finite_values, 99.5) < 200
        or np.percentile(finite_values, 0.5) > -500
    ):
        print(
            "\nADVERTENCIA: el rango no parece una TC en HU original. "
            "Verifica que el volumen no haya sido normalizado previamente."
        )



## 7. Inferencia congelada con monitor de GPU

La ejecución oficial usa:

```text
nnUNet_predict
-t 101
-m 3d_fullres
-f 0
-tr STUNetTrainer_small
-chk small_ep4k
--mode fast
--disable_tta
```

El monitor consulta `nvidia-smi` cada segundo. Esto mide la memoria del dispositivo durante el subproceso real de nnU-Net, no únicamente la memoria del kernel de Jupyter.


In [ ]:

# 7. Ejecutar inferencia y registrar VRAM/tiempo
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

def query_gpu_row() -> dict | None:
    if shutil.which("nvidia-smi") is None:
        return None

    cmd = [
        "nvidia-smi",
        "--query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw",
        "--format=csv,noheader,nounits",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    line = result.stdout.strip().splitlines()
    if not line:
        return None

    parts = [x.strip() for x in line[0].split(",")]
    if len(parts) < 7:
        return None

    return {
        "timestamp_nvidia": parts[0],
        "gpu_name": parts[1],
        "memory_used_mib": float(parts[2]),
        "memory_total_mib": float(parts[3]),
        "utilization_gpu_percent": float(parts[4]),
        "temperature_c": float(parts[5]),
        "power_w": float(parts[6].replace("[N/A]", "nan")),
    }

def write_gpu_csv(rows: list[dict], path: Path) -> None:
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

if not RUN_INFERENCE:
    print("RUN_INFERENCE=False. Cambia a True cuando el setup y el caso estén listos.")
else:
    if not PREPARED_INPUT.exists():
        raise FileNotFoundError(PREPARED_INPUT)

    predict_executable = shutil.which("nnUNet_predict")
    if predict_executable is None:
        raise RuntimeError(
            "No se encontró nnUNet_predict. Reinicia el kernel y verifica la instalación."
        )

    env = os.environ.copy()
    env["nnUNet_raw_data_base"] = str(RAW_BASE)
    env["nnUNet_preprocessed"] = str(PREPROCESSED)
    env["RESULTS_FOLDER"] = str(RESULTS_FOLDER)

    command = [
        predict_executable,
        "-i", str(INPUT_DIR),
        "-o", str(OUTPUT_DIR),
        "-t", TASK_ID,
        "-m", MODEL,
        "-f", "0",
        "-tr", TRAINER,
        "-chk", CHECKPOINT_NAME,
        "--mode", "fast",
        "--disable_tta",
    ]

    log_path = LOG_DIR / f"{CASE_ID}_inference.log"
    gpu_csv_path = LOG_DIR / f"{CASE_ID}_gpu.csv"
    run_json_path = LOG_DIR / f"{CASE_ID}_run.json"

    baseline = query_gpu_row()
    print("Comando:")
    print(" ".join(command))
    print("\nGPU inicial:")
    print(json.dumps(baseline, indent=2, ensure_ascii=False))

    start_wall = time.perf_counter()
    started_at = datetime.now(timezone.utc).isoformat()
    gpu_rows = []

    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            text=True,
        )

        while process.poll() is None:
            row = query_gpu_row()
            if row is not None:
                row["elapsed_s"] = time.perf_counter() - start_wall
                gpu_rows.append(row)
            time.sleep(1.0)

        return_code = process.wait()

    elapsed_s = time.perf_counter() - start_wall
    write_gpu_csv(gpu_rows, gpu_csv_path)

    peak_used = max(
        (row["memory_used_mib"] for row in gpu_rows),
        default=None,
    )
    mean_util = (
        sum(row["utilization_gpu_percent"] for row in gpu_rows) / len(gpu_rows)
        if gpu_rows else None
    )
    peak_temp = max(
        (row["temperature_c"] for row in gpu_rows),
        default=None,
    )
    baseline_used = baseline["memory_used_mib"] if baseline else None
    estimated_peak_delta = (
        peak_used - baseline_used
        if peak_used is not None and baseline_used is not None
        else None
    )

    run_summary = {
        "case_id": CASE_ID,
        "started_at_utc": started_at,
        "elapsed_s": elapsed_s,
        "return_code": return_code,
        "command": command,
        "prediction_path": str(PREDICTION_PATH),
        "prediction_exists": PREDICTION_PATH.exists(),
        "baseline_gpu": baseline,
        "peak_memory_used_mib": peak_used,
        "baseline_memory_used_mib": baseline_used,
        "estimated_peak_delta_mib": estimated_peak_delta,
        "mean_gpu_utilization_percent": mean_util,
        "peak_temperature_c": peak_temp,
        "gpu_samples": len(gpu_rows),
        "log_path": str(log_path),
        "gpu_csv_path": str(gpu_csv_path),
    }

    run_json_path.write_text(
        json.dumps(run_summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(json.dumps(run_summary, indent=2, ensure_ascii=False))

    if return_code != 0:
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
        print("\nÚltimas líneas del log:")
        print("\n".join(tail))
        raise RuntimeError(f"nnUNet_predict terminó con código {return_code}")

    if not PREDICTION_PATH.exists():
        raise FileNotFoundError(
            f"La inferencia terminó sin producir {PREDICTION_PATH}"
        )



## 8. Inspección de la salida renal

STU-Net-S fue preentrenada con las etiquetas de TotalSegmentator usadas por sus autores. En el mapa oficial de STU-Net:

- `38 = kidney_left`
- `39 = kidney_right`

La presencia de ambas clases no demuestra segmentación tumoral. Este piloto solo verifica transferencia anatómica directa y factibilidad computacional.


In [ ]:

# 8. Resumen de etiquetas y vista renal
from __future__ import annotations

import json
import numpy as np
import matplotlib.pyplot as plt

if not PREDICTION_PATH.exists():
    print("Aún no existe una predicción:", PREDICTION_PATH)
else:
    import nibabel as nib

    labels_path = PROJECT_ROOT / "label_orders.json"
    label_map = json.loads(labels_path.read_text(encoding="utf-8"))

    ct_img = nib.load(str(PREPARED_INPUT))
    seg_img = nib.load(str(PREDICTION_PATH))

    if ct_img.shape != seg_img.shape:
        raise ValueError(
            f"Geometría incompatible: TC={ct_img.shape}, predicción={seg_img.shape}"
        )

    ct = np.asarray(ct_img.dataobj, dtype=np.float32)
    seg = np.asarray(seg_img.dataobj, dtype=np.uint16)

    labels, counts = np.unique(seg, return_counts=True)
    table = []
    for label, count in zip(labels.tolist(), counts.tolist()):
        table.append({
            "label": int(label),
            "name": label_map.get(str(label), "unknown"),
            "voxels": int(count),
        })

    print("Etiquetas presentes:")
    for row in table:
        if row["label"] != 0:
            print(row)

    kidney_left = seg == 38
    kidney_right = seg == 39
    kidney_union = kidney_left | kidney_right

    renal_summary = {
        "kidney_left_present": bool(kidney_left.any()),
        "kidney_right_present": bool(kidney_right.any()),
        "kidney_left_voxels": int(kidney_left.sum()),
        "kidney_right_voxels": int(kidney_right.sum()),
    }
    print("\nResumen renal:")
    print(json.dumps(renal_summary, indent=2, ensure_ascii=False))

    if not kidney_union.any():
        print("\nNo se detectaron riñones. Revisa el log, la cobertura anatómica y los HU.")
    else:
        z_index = int(np.argmax(kidney_union.sum(axis=(0, 1))))
        ct_slice = np.rot90(ct[:, :, z_index])
        kidney_slice = np.rot90(kidney_union[:, :, z_index])

        low, high = np.percentile(ct_slice[np.isfinite(ct_slice)], [1, 99])

        plt.figure(figsize=(7, 7))
        plt.imshow(np.clip(ct_slice, low, high), cmap="gray")
        plt.contour(kidney_slice.astype(np.uint8), levels=[0.5])
        plt.title(f"{CASE_ID} — corte z={z_index}, contorno renal STU-Net-S")
        plt.axis("off")
        plt.tight_layout()
        preview_path = LOG_DIR / f"{CASE_ID}_kidney_preview.png"
        plt.savefig(preview_path, dpi=160, bbox_inches="tight")
        plt.show()
        print("Vista guardada en:", preview_path)



## 9. Manifiesto reproducible

El manifiesto registra versiones, hashes y métricas de ejecución. Debe conservarse junto con los resultados para comparar más adelante:

1. STU-Net-S FP32;
2. STU-Net-S rotada en FP32;
3. PTQ convencional;
4. TurboConv a igual precisión.


In [ ]:

# 9. Guardar manifiesto final
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str | None:
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def git_commit(repo: Path) -> str | None:
    if not (repo / ".git").exists():
        return None
    result = subprocess.run(
        ["git", "-C", str(repo), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=False,
    )
    return result.stdout.strip() or None

try:
    import torch
    torch_info = {
        "torch_version": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
except Exception as exc:
    torch_info = {"torch_import_error": repr(exc)}

run_json_path = LOG_DIR / f"{CASE_ID}_run.json"
run_summary = (
    json.loads(run_json_path.read_text(encoding="utf-8"))
    if run_json_path.exists()
    else None
)

manifest = {
    "experiment": "stunet_s_frozen_feasibility",
    "case_id": CASE_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(),
    "python": sys.version,
    "torch": torch_info,
    "stunet_git_commit": git_commit(REPO_DIR),
    "trainer": TRAINER,
    "checkpoint_name": CHECKPOINT_NAME,
    "checkpoint_path": str(FOLD_DIR / f"{CHECKPOINT_NAME}.model"),
    "checkpoint_sha256": sha256_file(FOLD_DIR / f"{CHECKPOINT_NAME}.model"),
    "input_path": str(PREPARED_INPUT),
    "input_sha256": sha256_file(PREPARED_INPUT),
    "prediction_path": str(PREDICTION_PATH),
    "prediction_sha256": sha256_file(PREDICTION_PATH),
    "run_summary": run_summary,
    "environment_variables": {
        "nnUNet_raw_data_base": os.environ.get("nnUNet_raw_data_base"),
        "nnUNet_preprocessed": os.environ.get("nnUNet_preprocessed"),
        "RESULTS_FOLDER": os.environ.get("RESULTS_FOLDER"),
    },
    "scientific_scope": {
        "fine_tuning": False,
        "training": False,
        "turbo_conv": False,
        "purpose": "computational feasibility and direct anatomical transfer only",
    },
}

manifest_path = LOG_DIR / f"{CASE_ID}_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Manifiesto:", manifest_path)
print(json.dumps(manifest, indent=2, ensure_ascii=False)[:5000])



## Criterios para aprobar este piloto

El experimento pasa a la siguiente fase cuando se cumplan todos los criterios:

1. `nnUNet_predict` termina con código `0`.
2. No ocurre `CUDA out of memory`.
3. La predicción conserva la geometría del volumen de entrada.
4. Aparece al menos una segmentación renal anatómicamente plausible.
5. El pico de VRAM y el tiempo total quedan registrados.
6. La vista renal confirma visualmente que el resultado no está desalineado.
7. El manifiesto y el log se guardan sin errores.

Una segmentación renal imperfecta **no invalida automáticamente la factibilidad**; debe distinguirse entre:

- error de dominio o calidad de imagen;
- cobertura anatómica insuficiente;
- fallo de preprocesamiento;
- límite real de memoria;
- fallo del modelo.

### Siguiente notebook

`02_stunet_s_turboconv_weight_sanity.ipynb`

Su objetivo será reproducir en los pesos de STU-Net-S el experimento previo de nnU-Net v1:

- PTQ uniforme;
- rotación Walsh–Hadamard;
- cuantización rotacional;
- MSE, error relativo, outliers y análisis por capa.

Todavía sin modificar el grafo funcional completo.
